# Python syntax pack — Pass the coding round

Cheat sheet from the **AI FDE / AI Engineering coding round** session.

**Rule:** most people fail because they are debugging Python *and* hunting for an algorithm. Separate the two. Drill this notebook until the syntax is automatic, then pattern-match.

Run top to bottom. Each section has a tiny assert so you know it stuck.


## 0. Imports you will type every round


In [1]:
from collections import defaultdict, Counter, deque
from typing import Sequence
import heapq
from heapq import nlargest


## 1. Pick the container, then write the code

| Container | Cost tell | Use when |
|---|---|---|
| `list` | O(1) index, **O(n) at the front** | ordered sequence, random access |
| `set` | O(1) membership, no duplicates | seen / visited |
| `dict` | O(1) lookup, insertion-ordered (3.7+) | maps, frequency tables |
| `tuple` | immutable; hashable if contents are | keys, points, heap entries |
| `deque` | O(1) left and right | queues, BFS, sliding removals |


In [2]:
nums = [1, 2, 3]      # O(1) index, O(n) at the front
seen = {1, 2, 3}      # O(1) membership, no duplicates
freq = {"a": 1}       # O(1) lookup, insertion ordered
point = (3, 4)        # immutable, hashable if its contents are

assert 2 in seen
assert freq["a"] == 1
assert point in {(3, 4)}  # tuple is hashable → can be a set/dict key


### A list is not a queue

`nums.pop(0)` is **O(n)** and will time out on a large input. Reach for a `deque` the moment anything is removed from the front.


In [3]:
from collections import deque

q = deque([1, 2, 3])
assert q.popleft() == 1          # O(1)
q.append(4)                      # O(1) right
q.appendleft(0)                  # O(1) left
assert list(q) == [0, 2, 3, 4]

# NEVER do this in a hot loop:
# nums.pop(0)  # O(n) each time → O(n²) overall


## 2. Loop without an index — slicing, enumerate, zip, comprehensions


In [4]:
s = "abcdef"
assert s[::-1] == "fedcba"   # reversed
assert s[1:4] == "bcd"       # end exclusive

nums = ["a", "b", "c"]
assert list(enumerate(nums)) == [(0, "a"), (1, "b"), (2, "c")]

xs, ys = [1, 2, 3], [10, 20, 30]
assert list(zip(xs, ys)) == [(1, 10), (2, 20), (3, 30)]

# index lookup in O(n) build, then O(1) query
lookup = {x: i for i, x in enumerate([10, 20, 30])}
assert lookup[20] == 1


### Build a grid with a comprehension — the aliasing trap

`[[0] * 3] * 2` makes **two references to one row**. Writing `grid[0][0]` then changes every row.


In [5]:
rows, cols = 2, 3

# WRONG — aliased rows
bad = [[0] * cols] * rows
bad[0][0] = 9
assert bad == [[9, 0, 0], [9, 0, 0]]  # both rows mutated

# RIGHT — fresh row each time
grid = [[0] * cols for _ in range(rows)]
grid[0][0] = 9
assert grid == [[9, 0, 0], [0, 0, 0]]


## 3. Dicts that never raise — `get`, `defaultdict`, `Counter`

Never write a `KeyError` on a frequency table. `Counter` and `defaultdict` remove the branch that guards the first insertion. `Counter` also gives you `most_common(k)` for free.


In [6]:
# counting with a default
freq = {}
for c in "aab":
    freq[c] = freq.get(c, 0) + 1
assert freq == {"a": 2, "b": 1}

# groups without an existence check
groups = defaultdict(list)
for word in ["eat", "tea", "tan", "nat"]:
    key = "".join(sorted(word))
    groups[key].append(word)
assert sorted(groups["".join(sorted("eat"))]) == ["eat", "tea"]

# Valid Anagram, one line
s, t = "anagram", "nagaram"
assert Counter(s) == Counter(t)

# Top-k frequencies for free
assert Counter("mississippi").most_common(2) == [("i", 4), ("s", 4)]


## 4. Heaps — default is a **min**-heap, so negate for max

Judge platforms lag Python versions. Prefer negate over `heappush_max`.

Tuple ordering is what makes Dijkstra and K Closest Points short: compare by first element, break ties later.


In [7]:
heap = []
for x in [5, 1, 3]:
    heapq.heappush(heap, x)       # O(log n)
assert heap[0] == 1               # peek, O(1) — always the minimum
assert heapq.heappop(heap) == 1   # O(log n)

# max-heap: push the negative
max_heap = []
for val in [5, 1, 3]:
    heapq.heappush(max_heap, -val)
assert -heapq.heappop(max_heap) == 5

# heapify is O(n), in place
nums = [5, 1, 3, 2]
heapq.heapify(nums)
assert nums[0] == 1

# Dijkstra / K Closest shape: (priority, payload)
dijk = []
heapq.heappush(dijk, (0.5, "node_a"))
heapq.heappush(dijk, (0.1, "node_b"))
assert heapq.heappop(dijk)[1] == "node_b"


## 5. Five traps that cost the round

| Trap | Why it fails | Fix |
|---|---|---|
| `s += c` in a loop | O(n²) string copies | append to a list, `"".join` once |
| `nums.pop(0)` | O(n) each call | `deque.popleft()` |
| `[[0]*n]*m` | aliased rows | `[[0]*n for _ in range(m)]` |
| `def f(x, acc=[])` | default shared across calls | `acc=None` then `acc = []` |
| Recursion past ~1000 | sys recursion limit | convert to iterative |


In [8]:
# string concat trap
chars = []
for c in "hello":
    chars.append(c)
assert "".join(chars) == "hello"

# mutable default trap
def wrong(x, acc=[]):
    acc.append(x)
    return acc

assert wrong(1) == [1]
assert wrong(2) == [1, 2]  # leaked state from previous call

def right(x, acc=None):
    if acc is None:
        acc = []
    acc.append(x)
    return acc

assert right(1) == [1]
assert right(2) == [2]


## 6. Input signal → pattern

Memorise the tell, not the LeetCode title.

| You hear… | Reach for… |
|---|---|
| "k largest", "k closest", a stream | heap of size k |
| Sorted array, find a pair | two pointers |
| Contiguous run, repairable constraint | sliding window |
| "Minimise the maximum" / "min capacity" | binary search on the answer |
| Dependencies, "order of" | topological sort |
| "Next greater", "previous smaller" | monotonic stack |
| Nesting, matching, parsing | plain stack |
| Grid of cells | BFS for shortest, DFS for reachability |


## 7. Template — sliding window

Both pointers only move forward → O(n) even with the inner `while`.

Problems: Longest Substring · Character Replacement · Minimum Window Substring · Permutation in String


In [9]:
def longest_valid_window(s: str, is_invalid) -> int:
    """Generic sliding window. Replace is_invalid with your constraint."""
    left = best = 0
    state: dict = {}

    for right in range(len(s)):
        state[s[right]] = state.get(s[right], 0) + 1

        while is_invalid(state):      # shrink until valid
            state[s[left]] -= 1
            if state[s[left]] == 0:
                del state[s[left]]
            left += 1

        best = max(best, right - left + 1)
    return best

# Example: at most 2 distinct characters
assert longest_valid_window("eceba", lambda st: len(st) > 2) == 3  # "ece"


### Live solve — Longest Substring Without Repeating (LC 3)

Every character enters and leaves the window at most once → O(n). Storing the last index instead of a set lets `left` jump past the duplicate rather than crawl.


In [10]:
def length_of_longest_substring(s: str) -> int:
    seen, left, best = set(), 0, 0
    for right, ch in enumerate(s):
        while ch in seen:
            seen.remove(s[left])
            left += 1
        seen.add(ch)
        best = max(best, right - left + 1)
    return best

assert length_of_longest_substring("abcabcbb") == 3
assert length_of_longest_substring("bbbbb") == 1
assert length_of_longest_substring("pwwkew") == 3
assert length_of_longest_substring("") == 0

# Jump-left variant (last index)
def length_of_longest_substring_jump(s: str) -> int:
    last, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1
        last[ch] = right
        best = max(best, right - left + 1)
    return best

assert length_of_longest_substring_jump("abba") == 2


## 8. Template — top k with a heap

Min-heap of size **k** for the **k largest**. O(n log k) time, O(k) space. Works on a stream that does not fit in memory (the usual follow-up).


In [11]:
def top_k(items, k: int):
    """items: iterable of (item, score). Returns k largest by score."""
    heap = []
    for item, score in items:
        heapq.heappush(heap, (score, item))
        if len(heap) > k:
            heapq.heappop(heap)     # evict the smallest
    return [item for score, item in sorted(heap, reverse=True)]

assert top_k([("a", 1), ("b", 5), ("c", 3), ("d", 4)], 2) == ["b", "d"]


### Live solve — Top K Frequent Elements (LC 347)

Three ways, one problem: **Sort** O(u log u) · **Heap** O(n log k) · **Buckets** O(n).

A frequency can never exceed n, so `n + 1` buckets index frequency directly and the sort disappears.


In [12]:
def top_k_frequent_sort(nums: list[int], k: int) -> list[int]:
    count = Counter(nums)
    return sorted(count, key=count.get, reverse=True)[:k]

def top_k_frequent_heap(nums: list[int], k: int) -> list[int]:
    count = Counter(nums)
    # nlargest over (freq, num); or size-k min-heap of (freq, num)
    return [num for num, _ in count.most_common(k)]

def top_k_frequent_buckets(nums: list[int], k: int) -> list[int]:
    count = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for num, freq in count.items():
        buckets[freq].append(num)
    out = []
    for freq in range(len(buckets) - 1, 0, -1):
        for num in buckets[freq]:
            out.append(num)
            if len(out) == k:
                return out
    return out

nums, k = [1, 1, 1, 2, 2, 3], 2
assert set(top_k_frequent_sort(nums, k)) == {1, 2}
assert set(top_k_frequent_heap(nums, k)) == {1, 2}
assert set(top_k_frequent_buckets(nums, k)) == {1, 2}


## 9. Template — BFS (mark on **push**)

Marking visited on pop lets the same cell enter the queue many times. That single line is the difference between O(V+E) and a queue that blows up on a dense grid.

Freeze the level with `for _ in range(len(queue))` when you need distance / layers.


In [13]:
def bfs_shortest(grid, starts):
    """grid: 0 empty / 1 blocked. starts: list of (r, c). Returns steps to visit all reachables (demo)."""
    rows, cols = len(grid), len(grid[0])
    def valid(r, c):
        return 0 <= r < rows and 0 <= c < cols and grid[r][c] == 0

    def neighbours(r, c):
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            yield r + dr, c + dc

    queue = deque(starts)
    visited = set(starts)          # mark on push
    steps = 0
    while queue:
        for _ in range(len(queue)):  # freeze the level
            r, c = queue.popleft()
            for nr, nc in neighbours(r, c):
                if valid(nr, nc) and (nr, nc) not in visited:
                    visited.add((nr, nc))
                    queue.append((nr, nc))
        if queue:
            steps += 1
    return steps, visited

grid = [
    [0, 0, 1],
    [0, 0, 0],
    [1, 0, 0],
]
steps, seen = bfs_shortest(grid, [(0, 0)])
assert (0, 0) in seen and (2, 2) in seen
assert (0, 2) not in seen  # blocked


## 10. Template — topological sort (Kahn)

Prerequisites are a directed graph. Without the length check, a cyclic graph returns a **partial** order that looks valid.

Problems: Course Schedule · Course Schedule II · Alien Dictionary


In [14]:
def topo_order(n: int, prerequisites: list[list[int]]) -> list[int]:
    graph, indegree = defaultdict(list), [0] * n
    for course, required in prerequisites:
        graph[required].append(course)
        indegree[course] += 1

    queue = deque(v for v in range(n) if indegree[v] == 0)
    order = []
    while queue:
        v = queue.popleft()
        order.append(v)
        for nxt in graph[v]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                queue.append(nxt)

    return order if len(order) == n else []   # short means a cycle

# 0←1, 0←2  (take 1 and 2 before 0) — edges as [course, required]
assert topo_order(3, [[0, 1], [0, 2]]) in ([1, 2, 0], [2, 1, 0])
assert topo_order(2, [[0, 1], [1, 0]]) == []  # cycle


## 11. Contest Python vs production Python

Correct output is the entry fee, not the score. AI FDE / AI Engineering rounds grade **failure behaviour**: types, docstring with complexity, explicit raise on invalid input, bounded concurrency.


In [15]:
# contest
def topk(c, k):
    return sorted(c, key=lambda x: -x[1])[:k]

# production shape
def top_k_chunks(chunks: Sequence[tuple[str, float]], k: int) -> list[tuple[str, float]]:
    """O(n log k) time, O(k) space."""
    if k < 0:
        raise ValueError("k must be non-negative")
    return nlargest(min(k, len(chunks)), chunks, key=lambda x: x[1])

chunks = [("a", 0.1), ("b", 0.9), ("c", 0.5)]
assert top_k_chunks(chunks, 2) == [("b", 0.9), ("c", 0.5)]

try:
    top_k_chunks(chunks, -1)
    assert False, "should have raised"
except ValueError:
    pass


### Async — the most common AI take-home mistake

`await` inside a `for` loop runs **one call at a time**. Prefer `asyncio.gather`, then bound concurrency, retry with jitter, and batch before you parallelise.


In [ ]:
# illustration only — do not run unbounded gather on 10k real API calls
import asyncio

async def embed(t: str) -> str:
    await asyncio.sleep(0.01)
    return t.upper()

async def sequential(texts):
    out = []
    for t in texts:
        out.append(await embed(t))  # one at a time
    return out

async def parallel(texts):
    return await asyncio.gather(*(embed(t) for t in texts))

async def bounded(texts, limit=2):
    sem = asyncio.Semaphore(limit)
    async def one(t):
        async with sem:
            return await embed(t)
    return await asyncio.gather(*(one(t) for t in texts))

texts = ["a", "b", "c", "d"]
# Jupyter already has a running loop — use await, not asyncio.run()
assert await parallel(texts) == ["A", "B", "C", "D"]
assert await bounded(texts, 2) == ["A", "B", "C", "D"]


## 12. Drill — blank the cells and re-type from memory

48 hours later, delete your solutions and re-solve from blank. That is the step that moves a pattern from *recognised* to *recallable*.

**Write from memory (no peeking):**
1. Grid comprehension that does **not** alias rows
2. Size-k min-heap for k largest
3. Sliding window skeleton with `left` / `right` / `state`
4. BFS with mark-on-push and level freeze
5. Kahn topo sort with cycle detection via `len(order) == n`

When those five are automatic, open Blind 75 / Neetcode 150 and only then hunt algorithms.


In [ ]:
# Scratch cell — re-type the five templates here from memory.
pass
